### Preprocessing Extensions
This notebook EXTENDS the validated pipeline from 01_preprocessing.ipynb
(unchanged, remains the source of truth for the original 2-class, 8-channel 
dataset at Data/processed/). Produces TWO additional, separate datasets, 
neither overwrites the original.

### Contents
- Channel renaming utility (copied, unchanged from original)
- preprocess_subject() (copied, unchanged from original)
- preprocess_subject_multiclass() (new, calls preprocess_subject() 
  twice with different run/event-id configs)
- Channel-expansion lateralisation check (new, validates candidate 
  16-channel set on full 106 subjects)
- run_multiclass_preprocessing() / run_16ch_preprocessing() (new, 
  loop over all subjects, call the respective preprocessing function, 
  save each subject's output as npz with incremental resume capability)

### Extension 1: Multiclass (4-class)
Adds runs R06, R10, R14 (both fists / both feet imagery) alongside the 
original R04, R08, R12 (left/right fist), producing a 4-class dataset:
- 0 = left fist, 1 = right fist, 2 = both fists, 3 = both feet
Output: Data/processed_multiclass/
Channels: same 8 as original (C3, Cz, C4, FC3, FCz, CP3, CPz, CP4)

### Extension 2: 16-channel expansion
Adds 8 new channels, selected via full-106-subject lateralisation 
validation (same T1/T2 variance-ratio method used to originally select/
exclude channels in 00_data_exploration.ipynb):
- New: CP5, CP6, P3, P4, C5, C6, P5, P6 (4 left/4 right pairs), 
- Full set: original 8 + new 8 = 16 channels
Output: Data/processed_16ch/
Task: same binary left/right fist classification as original

### Status
- Multiclass: complete, 106/106 subjects processed
- 16-channel: complete, 106/106 subjects processed
- Both verified against original pipeline's output patterns (trial 
  counts, rejection rates) before full 106-subject runs

### Initial Setup

In [4]:
import mne
from mne.preprocessing import ICA

import torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import random
import re

from tqdm import tqdm
import os

mne.set_log_level("WARNING")

# Reproducibility
np.random.seed(25)
random.seed(25)
torch.manual_seed(25)

# Paths
data_path = Path("../Data/raw/physionet_mi_data/eeg-motor-movementimagery-dataset")
processed_path = Path("../Data/processed")
multiclass_path = Path("../Data/processed_multiclass")
multiclass_path.mkdir(exist_ok=True)

out_path = Path("../Data/processed_16ch")
out_path.mkdir(exist_ok=True)

In [5]:
# dataset parameters
# runs for binary classification (left vs right hand)
# for multiclass: extend to include R06,R10,R14 for feet and both fists
runs = ['R04', 'R08', 'R12']
event_id = {'T1': 2, 'T2': 3}
n_samples = 480
sfreq = 160.0
tmax = (n_samples - 1) / sfreq
tmin = 0.0

# preprocessing parameters decided during exploration
notch_freq = 60.0
bandpass_low = 8.0
bandpass_high = 30.0
ica_broad_low = 1.0
ica_broad_high = 79.0
ica_n_components = 20
ica_eog_threshold = 3.0

artifact_relative_multiplier = 3.0

# channel selection decided during exploration
# FC4 excluded due to incorrect lateralisation direction
selected_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
                      'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

# subjects
all_subjects = [f"S{i:03d}" for i in range(1, 110)]
excluded = ['S088', 'S092', 'S100']
valid_subjects = [s for s in all_subjects if s not in excluded]

In [3]:
print(f"MNE version: {mne.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data path exists: {data_path.exists()}")
print(f"Processed path exists: {processed_path.exists()}")
print(f"Valid subjects: {len(valid_subjects)}")
print(f"Selected channels: {selected_channels}")

MNE version: 1.12.1
PyTorch version: 2.11.0
Data path exists: True
Processed path exists: True
Valid subjects: 106
Selected channels: ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4', 'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']


### Channel renaming utility function

In [6]:
# reused across all preprocessing steps
# strips trailing dots and fixes capitalisation to match standard 1020 montage

montage = mne.channels.make_standard_montage('standard_1020')
montage_lookup = {ch.upper(): ch for ch in montage.ch_names}

def rename_channels_standard(raw):
    # strip trailing dots first
    rename_map = {ch: ch.rstrip('.') for ch in raw.ch_names}
    raw.rename_channels(rename_map)

    # fix capitalisation to match standard montage
    cap_fix = {}
    for ch in raw.ch_names:
        upper = ch.upper()
        cap_fix[ch] = montage_lookup[upper] if upper in montage_lookup else ch
    raw.rename_channels(cap_fix)
    raw.set_montage(montage, on_missing='ignore')

    return raw


In [5]:
print("Channel renaming utility function defined")
print(f"Montage loaded: {len(montage.ch_names)} channels")

Channel renaming utility function defined
Montage loaded: 94 channels


### Single subject preprocessing function

In [7]:
def preprocess_subject(subject_id):
    """
    Full preprocessing pipeline for one subject.
    Returns clean epochs ready for model training.
    Steps: notch, bandpass, CAR, ICA EOG removal, epoching,
    hybrid threshold rejection, baseline correction,
    channel selection, z-score normalisation.
    """

    # load all three runs
    raw_list_narrow = []
    raw_list_broad = []

    for run in runs:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"

        # narrow filtered version for analysis (8-30 Hz)
        raw_narrow = mne.io.read_raw_edf(edf_path, preload=True)
        raw_narrow.notch_filter(freqs=notch_freq)
        raw_narrow.filter(
            l_freq=bandpass_low,
            h_freq=bandpass_high,
            method='fir',
            fir_design='firwin'
        )
        raw_narrow.set_eeg_reference('average', projection=False)
        raw_list_narrow.append(raw_narrow)

        # broad filtered version for ICA fitting (1-79 Hz)
        raw_broad = mne.io.read_raw_edf(edf_path, preload=True)
        raw_broad.notch_filter(freqs=notch_freq)
        raw_broad.filter(
            l_freq=ica_broad_low,
            h_freq=ica_broad_high,
            method='fir',
            fir_design='firwin'
        )
        raw_broad.set_eeg_reference('average', projection=False)
        raw_list_broad.append(raw_broad)

    # concatenate runs
    raw_narrow_concat = mne.concatenate_raws(raw_list_narrow)
    raw_broad_concat = mne.concatenate_raws(raw_list_broad)

    # rename channels for both versions
    raw_narrow_concat = rename_channels_standard(raw_narrow_concat)
    raw_broad_concat = rename_channels_standard(raw_broad_concat)

    # fit ICA on broad filtered signal
    ica = ICA(
        n_components=ica_n_components,
        random_state=25,
        method='infomax',
        fit_params=dict(extended=True)
    )
    ica.fit(raw_broad_concat)

    # find and remove EOG components using frontal channels as proxy
    eog_indices, _ = ica.find_bads_eog(
        raw_broad_concat,
        ch_name=['Fp1', 'Fp2'],
        threshold=ica_eog_threshold
    )
    ica.exclude = eog_indices

    # apply ICA to narrow filtered signal
    ica.apply(raw_narrow_concat)

    # extract epochs
    # binary classification: T1 left fist vs T2 right fist
    # for multiclass extension: add T3 (both fists) and T4 (both feet)
    # runs R06, R10, R14 contain feet imagery, R05, R09, R13 contain both fists
    # event_id would become {'T1': 2, 'T2': 3, 'T3': 2, 'T4': 3} on different runs
    # selected runs would also need updating from R04, R08, R12
    events, _ = mne.events_from_annotations(raw_narrow_concat)
    epochs = mne.Epochs(
        raw_narrow_concat,
        events,
        event_id=event_id,
        tmin=tmin,
        tmax=tmax,
        baseline=None,
        preload=True
    )

    # relative and IQR threshold rejection, artifact rejection using subject level thresholds
    """ a fixed amplitude limit of 250 uV was tested during exploration, but caused complete exclusion of high amplitude subjects as S009, S013, S024, S109 all had natural amplitudes above 250 uV, replaced with subject level relative and IQR thresholds. Where relative threshold (3x subject median) sets upper bound based on typical amplitude and IQR threshold (median + 3x IQR) catches outliers within subject's own distribution"""
    data = epochs.get_data()
    ptp = (data.max(axis=2) - data.min(axis=2)).max(axis=1) * 1e6
    subject_median = np.median(ptp)

    # relative threshold: 3x subject median
    relative_threshold = subject_median * artifact_relative_multiplier

    # IQR based outlier detection: median + 3x IQR
    q75 = np.percentile(ptp, 75)
    q25 = np.percentile(ptp, 25)
    iqr = q75 - q25
    iqr_threshold = subject_median + (3 * iqr)

    keep_mask = (ptp <= relative_threshold) & (ptp <= iqr_threshold)
    epochs = epochs[keep_mask]

    # baseline correction removed
    # bandpass filter at 8-30 Hz removes frequencies below 8 Hz
    # covering the same low frequency contamination baseline correction addresses
    # epochs.apply_baseline((0, 0.1))

    # channel selection
    epochs.pick_channels(selected_channels)

    # z-score normalisation per channel
    data = epochs.get_data()
    mean = data.mean(axis=(0, 2), keepdims=True)
    std = data.std(axis=(0, 2), keepdims=True)
    data_normalised = (data - mean) / (std + 1e-8)

    return data_normalised, epochs.events[:, 2], len(eog_indices), keep_mask.sum()

In [7]:
data, labels, n_eog, n_kept = preprocess_subject('S001')
print(f"Unique label values: {set(labels)}")
print(f"Label counts: {[(l, (labels==l).sum()) for l in set(labels)]}")
print(f"Data shape: {data.shape}")

Unique label values: {np.int64(2), np.int64(3)}
Label counts: [(np.int64(2), np.int64(22)), (np.int64(3), np.int64(22))]
Data shape: (44, 16, 480)


### Multi class function

In [8]:
def preprocess_subject_multiclass(subject_id, n_classes=4, channel_list=None):
    """
    calls existing preprocess_subject() twice with different run/event_id
    configs to build full 4-class dataset, reusing all validated
    ICA/rejection/normalisation logic unchanged

    channel_list: explicit list of channels to use, decoupled from
    whatever the global selected_channels happens to be set to.
    Defaults to original 8 if not specified (preserves all existing
    behavior/calls unchanged). Pass the 16-channel list explicitly
    to generate the combined multiclass+16ch variant.
    """
    global runs, event_id, selected_channels

    if channel_list is None:
        channel_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4']

    try:
        selected_channels = channel_list
        runs = ['R04', 'R08', 'R12']
        event_id = {'T1': 2, 'T2': 3}
        data_lr, labels_lr, _, _ = preprocess_subject(subject_id)
        labels_lr_mapped = np.where(labels_lr == 2, 0, 1)

        if n_classes == 2:
            return data_lr, labels_lr_mapped

        selected_channels = channel_list
        runs = ['R06', 'R10', 'R14']
        event_id = {'T1': 2, 'T2': 3}
        data_ff, labels_ff, _, _ = preprocess_subject(subject_id)
        labels_ff_mapped = np.where(labels_ff == 2, 2, 3)

        data_all = np.concatenate([data_lr, data_ff], axis=0)
        labels_all = np.concatenate([labels_lr_mapped, labels_ff_mapped], axis=0)

        return data_all, labels_all

    finally:
        runs = ['R04', 'R08', 'R12']
        event_id = {'T1': 2, 'T2': 3}
        selected_channels = channel_list

In [9]:
# data_4class, labels_4class = preprocess_subject_multiclass('S001', n_classes=4)
# print(f"4-class data shape: {data_4class.shape}")
# print(f"4-class label values: {set(labels_4class)}")
# print(f"4-class label counts: {[(l, (labels_4class==l).sum()) for l in sorted(set(labels_4class))]}")

In [10]:
# test_subjects = ['S002', 'S010', 'S050', 'S090']

# for subj in test_subjects:
#     data, labels = preprocess_subject_multiclass(subj, n_classes=4)
#     counts = [(l, (labels==l).sum()) for l in sorted(set(labels))]
#     print(f"{subj}: shape {data.shape}, counts {counts}")

### 8 channel - run_multiclass_preprocessing

In [9]:
def run_multiclass_preprocessing(valid_subjects, n_classes=4, resume=True):
    """
    runs preprocess_subject_multiclass across all valid subjects
    saves each subject's data/labels as npz to multiclass_path
    (defined in parameters cell), original processed_path untouched
    """
    failed_subjects = []

    for idx, subject_id in enumerate(valid_subjects):
        npz_path = multiclass_path / f"{subject_id}_multiclass.npz"

        if resume and npz_path.exists():
            print(f"Skipping {subject_id} (already processed)")
            continue

        try:
            data, labels = preprocess_subject_multiclass(subject_id, n_classes=n_classes)
            np.savez(npz_path, data=data, labels=labels)
            print(f"{subject_id} ({idx+1}/{len(valid_subjects)}): shape {data.shape}, "
                  f"counts {[(l, (labels==l).sum()) for l in sorted(set(labels))]}")
        except Exception as e:
            print(f"FAILED {subject_id}: {e}")
            failed_subjects.append(subject_id)

    print(f"\nDone. {len(valid_subjects) - len(failed_subjects)}/{len(valid_subjects)} subjects processed successfully.")
    if failed_subjects:
        print(f"Failed subjects: {failed_subjects}")

    return failed_subjects

In [12]:
test_failed = run_multiclass_preprocessing(valid_subjects[:5], n_classes=4, resume=True)

Skipping S001 (already processed)
Skipping S002 (already processed)
Skipping S003 (already processed)
Skipping S004 (already processed)
Skipping S005 (already processed)

Done. 5/5 subjects processed successfully.


In [13]:
all_failed = run_multiclass_preprocessing(valid_subjects, n_classes=4, resume=True)

Skipping S001 (already processed)
Skipping S002 (already processed)
Skipping S003 (already processed)
Skipping S004 (already processed)
Skipping S005 (already processed)
Skipping S006 (already processed)
Skipping S007 (already processed)
Skipping S008 (already processed)
Skipping S009 (already processed)
Skipping S010 (already processed)
Skipping S011 (already processed)
Skipping S012 (already processed)
Skipping S013 (already processed)
Skipping S014 (already processed)
Skipping S015 (already processed)
Skipping S016 (already processed)
Skipping S017 (already processed)
Skipping S018 (already processed)
Skipping S019 (already processed)
Skipping S020 (already processed)
Skipping S021 (already processed)
Skipping S022 (already processed)
Skipping S023 (already processed)
Skipping S024 (already processed)
Skipping S025 (already processed)
Skipping S026 (already processed)
Skipping S027 (already processed)
Skipping S028 (already processed)
Skipping S029 (already processed)
Skipping S030 

### Channel expansion check

In [12]:
def channel_lateralization_check(subject_id, channels_to_test):
    """
    computes T1/T2 variance ratio for each candidate channel
    using same methodology as original FC4 validation check
    returns dict of channel -> ratio
    """
    raw_list = []
    for run in ['R04', 'R08', 'R12']:
        edf_path = data_path / subject_id / f"{subject_id}{run}.edf"
        raw_s = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
        raw_s.notch_filter(freqs=60, verbose=False)
        raw_s.filter(l_freq=8.0, h_freq=30.0, method='fir',
                    fir_design='firwin', verbose=False)
        raw_s.set_eeg_reference('average', projection=False, verbose=False)
        raw_list.append(raw_s)

    raw_concat = mne.concatenate_raws(raw_list, verbose=False)
    raw_concat = rename_channels_standard(raw_concat)

    available = [ch for ch in channels_to_test if ch in raw_concat.ch_names]

    events, _ = mne.events_from_annotations(raw_concat, verbose=False)
    epochs = mne.Epochs(raw_concat, events, event_id={'T1': 2, 'T2': 3},
                        tmin=0, tmax=3, baseline=None, preload=True, verbose=False)

    ratios = {}
    for ch in available:
        ch_data = epochs.get_data(picks=[ch])
        labels = epochs.events[:, -1]
        t1_var = ch_data[labels == 2].var()
        t2_var = ch_data[labels == 3].var()
        ratios[ch] = t1_var / t2_var if t2_var > 0 else np.nan

    return ratios

In [15]:
existing_8 = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4']

sample_edf = data_path / 'S001' / 'S001R04.edf'
sample_raw = mne.io.read_raw_edf(sample_edf, preload=False, verbose=False)
sample_raw = rename_channels_standard(sample_raw)
all_channels = sample_raw.ch_names

candidate_channels = [ch for ch in all_channels if ch not in existing_8]
print(f"Testing {len(candidate_channels)} candidate channels across 106 subjects")

results = {ch: [] for ch in candidate_channels}
failed = []

for idx, subject_id in enumerate(valid_subjects):
    try:
        ratios = channel_lateralization_check(subject_id, candidate_channels)
        for ch, r in ratios.items():
            if not np.isnan(r):
                results[ch].append(r)
        if (idx + 1) % 10 == 0:
            print(f"Processed {idx+1}/{len(valid_subjects)} subjects")
    except Exception as e:
        print(f"FAILED {subject_id}: {e}")
        failed.append(subject_id)

print(f"\nDone. {len(valid_subjects)-len(failed)}/{len(valid_subjects)} subjects succeeded.")
if failed:
    print(f"Failed: {failed}")
print(f"\nSample result counts: {[(ch, len(results[ch])) for ch in list(results.keys())[:5]]}")

Testing 56 candidate channels across 106 subjects
Processed 10/106 subjects
Processed 20/106 subjects
Processed 30/106 subjects
Processed 40/106 subjects
Processed 50/106 subjects
Processed 60/106 subjects
Processed 70/106 subjects
Processed 80/106 subjects
Processed 90/106 subjects
Processed 100/106 subjects

Done. 106/106 subjects succeeded.

Sample result counts: [('FC5', 106), ('FC1', 106), ('FC2', 106), ('FC4', 106), ('FC6', 106)]


In [16]:
summary_rows = []
for ch, ratios in results.items():
    ratios_arr = np.array(ratios)
    mean_ratio = ratios_arr.mean()
    std_ratio = ratios_arr.std()
    # "good" lateralization: ratio should differ meaningfully from 1.0
    # and be consistent (low std) across subjects
    distance_from_1 = abs(mean_ratio - 1.0)
    summary_rows.append({
        'channel': ch,
        'mean_ratio': mean_ratio,
        'std_ratio': std_ratio,
        'distance_from_1': distance_from_1,
        'n': len(ratios)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('distance_from_1', ascending=False)
print(summary_df.head(20).to_string(index=False))

channel  mean_ratio  std_ratio  distance_from_1   n
     F7    1.116159   0.191652         0.116159 106
    CP6    0.908278   0.170152         0.091722 106
     P3    1.088684   0.186831         0.088684 106
    AF7    1.085421   0.176579         0.085421 106
    CP5    1.082521   0.168670         0.082521 106
    FT7    1.076501   0.256804         0.076501 106
     P5    1.076490   0.174360         0.076490 106
     P4    0.925382   0.150791         0.074618 106
    PO3    1.070227   0.209178         0.070227 106
     P6    0.931933   0.151225         0.068067 106
     F5    1.067755   0.155295         0.067755 106
     P1    1.065387   0.175998         0.065387 106
    CP1    1.064891   0.153872         0.064891 106
     C6    0.935497   0.183070         0.064503 106
    AF8    0.937439   0.137878         0.062561 106
    PO7    1.060376   0.203227         0.060376 106
     F8    0.941321   0.137010         0.058679 106
     C5    1.058393   0.175598         0.058393 106
    CP2    0

In [17]:
selected_channels = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
                      'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

test_subjects_16ch = valid_subjects[:5] + valid_subjects[20:25] + valid_subjects[50:55] + valid_subjects[-5:]
# spread across early/mid/late subject IDs, similar to original exploration approach

for subj in test_subjects_16ch:
    data, labels, n_eog, n_kept = preprocess_subject(subj)
    print(f"{subj}: shape {data.shape}, n_kept {n_kept}, n_eog_removed {n_eog}")

S001: shape (44, 16, 480), n_kept 44, n_eog_removed 1
S002: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S003: shape (44, 16, 480), n_kept 44, n_eog_removed 3
S004: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S005: shape (44, 16, 480), n_kept 44, n_eog_removed 1
S021: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S022: shape (44, 16, 480), n_kept 44, n_eog_removed 2
S023: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S024: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S025: shape (44, 16, 480), n_kept 44, n_eog_removed 2
S051: shape (44, 16, 480), n_kept 44, n_eog_removed 2
S052: shape (45, 16, 480), n_kept 45, n_eog_removed 1
S053: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S054: shape (44, 16, 480), n_kept 44, n_eog_removed 2
S055: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S105: shape (45, 16, 480), n_kept 45, n_eog_removed 2
S106: shape (44, 16, 480), n_kept 44, n_eog_removed 2
S107: shape (45, 16, 480), n_kept 45, n_eog_removed 1
S108: shape (45, 16, 480), n

### run_16ch_preprocessing

In [13]:
def run_16ch_preprocessing(valid_subjects, resume=True):
    failed_subjects = []

    for idx, subject_id in enumerate(valid_subjects):
        npz_path = out_path / f"{subject_id}_16ch.npz"

        if resume and npz_path.exists():
            print(f"Skipping {subject_id} (already processed)")
            continue

        try:
            data, labels, n_eog, n_kept = preprocess_subject(subject_id)
            labels_remapped = np.where(labels == 2, 0, 1)  # 2(T1/left)->0, 3(T2/right)->1
            np.savez(npz_path, data=data, labels=labels_remapped)
            print(f"{subject_id} ({idx+1}/{len(valid_subjects)}): shape {data.shape}, n_kept {n_kept}")
        except Exception as e:
            print(f"FAILED {subject_id}: {e}")
            failed_subjects.append(subject_id)

    print(f"\nDone. {len(valid_subjects)-len(failed_subjects)}/{len(valid_subjects)} succeeded.")
    if failed_subjects:
        print(f"Failed: {failed_subjects}")
    return failed_subjects

In [29]:
all_failed_16ch = run_16ch_preprocessing(valid_subjects, resume=True)

S001 (1/106): shape (44, 16, 480), n_kept 44
S002 (2/106): shape (45, 16, 480), n_kept 45
S003 (3/106): shape (44, 16, 480), n_kept 44
S004 (4/106): shape (45, 16, 480), n_kept 45
S005 (5/106): shape (44, 16, 480), n_kept 44
S006 (6/106): shape (45, 16, 480), n_kept 45
S007 (7/106): shape (45, 16, 480), n_kept 45
S008 (8/106): shape (45, 16, 480), n_kept 45
S009 (9/106): shape (45, 16, 480), n_kept 45
S010 (10/106): shape (44, 16, 480), n_kept 44
S011 (11/106): shape (42, 16, 480), n_kept 42
S012 (12/106): shape (42, 16, 480), n_kept 42
S013 (13/106): shape (45, 16, 480), n_kept 45
S014 (14/106): shape (45, 16, 480), n_kept 45
S015 (15/106): shape (45, 16, 480), n_kept 45
S016 (16/106): shape (44, 16, 480), n_kept 44
S017 (17/106): shape (45, 16, 480), n_kept 45
S018 (18/106): shape (45, 16, 480), n_kept 45
S019 (19/106): shape (45, 16, 480), n_kept 45
S020 (20/106): shape (43, 16, 480), n_kept 43
S021 (21/106): shape (45, 16, 480), n_kept 45
S022 (22/106): shape (44, 16, 480), n_kept 

### 16 channel regeneration npz files - run_multiclass_16ch_preprocessing

In [14]:
def run_multiclass_16ch_preprocessing(valid_subjects, resume=True):
    out_path = Path("../Data/processed_multiclass_16ch")
    out_path.mkdir(exist_ok=True)

    ch16_list = ['C3', 'Cz', 'C4', 'FC3', 'FCz', 'CP3', 'CPz', 'CP4',
                 'CP5', 'CP6', 'P3', 'P4', 'C5', 'C6', 'P5', 'P6']

    failed_subjects = []

    for idx, subject_id in enumerate(valid_subjects):
        npz_path = out_path / f"{subject_id}_multiclass_16ch.npz"

        if resume and npz_path.exists():
            print(f"Skipping {subject_id} (already processed)")
            continue

        try:
            data, labels = preprocess_subject_multiclass(subject_id, n_classes=4, channel_list=ch16_list)
            labels_remapped = labels  # already correctly mapped 0-3 inside preprocess_subject_multiclass
            np.savez(npz_path, data=data, labels=labels_remapped)
            print(f"{subject_id} ({idx+1}/{len(valid_subjects)}): shape {data.shape}, "
                  f"counts {[(l, (labels==l).sum()) for l in sorted(set(labels))]}")
        except Exception as e:
            print(f"FAILED {subject_id}: {e}")
            failed_subjects.append(subject_id)

    print(f"\nDone. {len(valid_subjects)-len(failed_subjects)}/{len(valid_subjects)} succeeded.")
    if failed_subjects:
        print(f"Failed: {failed_subjects}")
    return failed_subjects

In [ ]:
# test_failed = run_multiclass_16ch_preprocessing(['S001', 'S002', 'S003'], resume=True)

Skipping S001 (already processed)
Skipping S002 (already processed)
Skipping S003 (already processed)

Done. 3/3 succeeded.


In [17]:
all_failed_mc16ch = run_multiclass_16ch_preprocessing(valid_subjects, resume=True)

S001 (1/106): shape (87, 16, 480), counts [(np.int64(0), np.int64(22)), (np.int64(1), np.int64(22)), (np.int64(2), np.int64(21)), (np.int64(3), np.int64(22))]
S002 (2/106): shape (88, 16, 480), counts [(np.int64(0), np.int64(23)), (np.int64(1), np.int64(22)), (np.int64(2), np.int64(22)), (np.int64(3), np.int64(21))]
S003 (3/106): shape (89, 16, 480), counts [(np.int64(0), np.int64(23)), (np.int64(1), np.int64(21)), (np.int64(2), np.int64(21)), (np.int64(3), np.int64(24))]
S004 (4/106): shape (90, 16, 480), counts [(np.int64(0), np.int64(23)), (np.int64(1), np.int64(22)), (np.int64(2), np.int64(22)), (np.int64(3), np.int64(23))]
S005 (5/106): shape (87, 16, 480), counts [(np.int64(0), np.int64(20)), (np.int64(1), np.int64(24)), (np.int64(2), np.int64(22)), (np.int64(3), np.int64(21))]
S006 (6/106): shape (90, 16, 480), counts [(np.int64(0), np.int64(24)), (np.int64(1), np.int64(21)), (np.int64(2), np.int64(22)), (np.int64(3), np.int64(23))]
S007 (7/106): shape (90, 16, 480), counts [(np

In [18]:
print(f"Failed: {all_failed_mc16ch}")

Failed: []
